In [0]:
from pyspark.sql import functions as F, types as T,window as W

### Section 1 — Read data from the Bronze table into a DataFrame

In [0]:
SOURCE_TABLE='operation.bronze.crm_sales_details'
TARGET_TABLE='operation.SILVER.crm_sales'

df_bronze=spark.table(SOURCE_TABLE)
print(f'''Load {SOURCE_TABLE} compeletly :''',df_bronze.count())
df_bronze.printSchema()

df_bronze.show(5)

### Section 2 — Analyze data quality using SQL (list all identified issues)

In [0]:
# DQ check 1 — duplicates on the business key , shown separately
display(df_bronze.groupBy("sls_ord_num","sls_prd_key").count().filter(F.col('count')>1).sort(F.col('count'),ascending=False))

print(f''' Business keys are (sls_ord_num,sls_prd_key).''')



In [0]:
# DQ check 2 — nulls per column
Nulls_count=df_bronze.select ([F.sum(F.trim(F.col(c)).isNull().cast("int")).alias(c) for c in df_bronze.columns])
display(Nulls_count)

# DQ check 3 — leading/trailing whitespace in string columns
for c in ['sls_ord_num','sls_prd_key','sls_cust_id']:
    n = df_bronze.filter(F.col(c) != F.trim(F.col(c))).count()
    print(f'Found {n} rows with leading or trailing whitespace in column {c}')

# DQ check 4 — date format & range: is the string parseable as yyyy-MM-dd, and is the range plausible?
for c in ['sls_order_dt','sls_ship_dt','sls_due_dt']:
    unparseable=df_bronze.filter(
        F.col(c).isNotNull()
        & F.try_to_date(F.trim(F.col(c)), 'yyyyMMdd').isNull()
    
        )
    print(f'''{c} values not parseable as yyyy-MM-dd: {unparseable.count()}''')
display(df_bronze.select(F.col('sls_order_dt')).distinct())

### Section 2.1 — Transform data (fix issues one by one)
### Each step fixes one issue and displays the result before moving to the next.

In [0]:
df_clean=(df_bronze
          .select(
              F.trim(F.col('sls_ord_num')).alias('sales_order_number'),
              F.trim(F.col('sls_prd_key')).alias('sales_product_key'),
              F.trim(F.col('sls_cust_id')).alias('sales_customer_id'),
              F.try_to_date(F.trim(F.col('sls_order_dt')), 'yyyyMMdd').alias('sales_order_date'),
              F.try_to_date(F.trim(F.col('sls_ship_dt')), 'yyyyMMdd').alias('sales_ship_date'),
              F.try_to_date(F.trim(F.col('sls_due_dt')), 'yyyyMMdd').alias('sales_due_date'),
              F.trim(F.col('sls_sales')).cast('double').alias('sales'),
              F.trim(F.col('sls_quantity')).cast('double').alias('sales_quantity'),
              F.trim(F.col('sls_price')).cast('double').alias('sales_price')
          ))
display(df_clean.limit(5))

Section 3 — Write the DataFrame to a new Silver table (friendly name)

In [0]:
df_clean.write.mode('overwrite').saveAsTable(TARGET_TABLE)
print(f'Wrote {df_clean.count():,} rows to {TARGET_TABLE}')